# Primera ley: trabajo, calor, trayectorias y calorimetría

**Introducción a la Termodinámica (4603003) · Licenciatura en Biología Molecular · UAM Cuajimalpa**  
**TermoIA-UAMC · TIA-NB01 · versión 1.2 · uso recomendado: semanas 2–4**

## Qué deberías poder hacer al terminar
1. Aplicar $\Delta U = q + w$ con una convención de signos explícita y detectar cuándo una fuente usa otra.
2. Distinguir funciones de estado ($U$, $H$) de cantidades que dependen de la trayectoria ($q$, $w$), y demostrarlo numéricamente.
3. Leer y construir diagramas $P$–$V$: identificar procesos, interpretar áreas como trabajo y analizar ciclos.
4. Plantear balances de energía en calorimetría, incluyendo la capacidad del calorímetro, cambios de fase y pérdidas de calor.
5. Pasar de $\Delta U$ (bomba calorimétrica, volumen constante) a $\Delta H$ (presión constante) y juzgar cuándo la diferencia importa.
6. Evaluar críticamente un resultado: unidades, orden de magnitud, supuestos del modelo e incertidumbre.

## Cómo trabajar este cuaderno
Este cuaderno **no** se evalúa por los números que imprime, sino por tu razonamiento. Es un cuaderno modular: **no se espera completar todas las secciones en una sola sesión ni necesariamente durante el mismo trimestre**; el docente indicará la ruta y los apartados evaluables.

- **Predice antes de ejecutar.** Muchas celdas tienen variables `pred_... = None`. Escribe ahí tu predicción *antes* de ejecutar la celda de cálculo. Mientras la predicción sea `None`, la celda oculta el resultado. No corrijas la predicción después de ver el resultado: una predicción equivocada y bien analizada vale lo mismo que una correcta.
- **Contrasta.** Después de cada predicción hay una celda de texto **Contraste**. Escribe en 2–4 líneas si el resultado confirmó o corrigió tu idea y, sobre todo, *por qué*.
- **Tu turno.** Las celdas con `# TODO` requieren que escribas código. Las llamadas a `verifica(...)` comprueban consistencia física (por ejemplo, que $\Delta U$ de un ciclo sea cero); no te dan la respuesta.
- Las secciones marcadas **★ Ampliación** son opcionales o para una sesión posterior.

## Convenciones (léelas: son la fuente de error más común)
- Primera ley: $\Delta U = q + w$, donde $q$ es el calor **recibido** por el sistema y $w$ es el trabajo hecho **sobre** el sistema (convención IUPAC; es la de Atkins y la mayoría de los textos de química y bioquímica).
- Trabajo de expansión–compresión: $w = -\int P_\text{ext}\,dV$. Si el gas se expande, $w<0$.
- Muchos textos de física e ingeniería escriben $\Delta U = Q - W$ con $W$ hecho **por** el sistema. Es la misma física con otro signo. Al leer cualquier fuente, identifica primero su convención.
- Unidades: presión en **kPa** y volumen en **L**. Observa que $1\ \text{kPa·L} = 10^{3}\ \text{Pa}\times10^{-3}\ \text{m}^3 = 1\ \text{J}$. Así, $R = 8.314\ \text{J/(mol·K)} = 8.314\ \text{kPa·L/(mol·K)}$ y las áreas en un diagrama $P$(kPa)–$V$(L) se leen directamente en joules.

## Rutas de trabajo y accesibilidad

Este cuaderno es **modular**. La programación es una herramienta para hacer visibles las consecuencias de un modelo; no sustituye el razonamiento termodinámico.

**Ruta recomendada por semana**

- **Semana 2 — variables de estado y modelos:** diagnóstico (§0), calentamiento isocórico (§1) y lectura de trayectorias \(P\)–\(V\) (§3.1–3.4).
- **Semana 3 — primera ley, trabajo y trayectoria:** §4 y cacería de errores (§6). Los ciclos de §5 pueden usarse como extensión.
- **Semana 4 — entalpía y calorimetría:** §2, §7.1 y §7.3, en coordinación con la práctica P01. §7.2, §7.4–7.6 son ampliaciones.

**Alternativas DUA.** Cuando el objetivo sea conceptual y no de programación, el docente puede proporcionar la salida numérica o una gráfica ya generada. En ese caso debes entregar la predicción, el planteamiento, la interpretación física y la verificación independiente. Cuando el objetivo explícito sea computacional, sí se evaluará el código correspondiente.

### Ciclo PIVED en este notebook

1. **Predecir:** registra una expectativa antes de ejecutar.
2. **Interrogar:** cuando se indique, pide a una IA que critique o explique tu planteamiento **después** de haber producido una primera respuesta propia.
3. **Verificar:** comprueba afirmaciones con unidades, derivación, cálculo, caso límite, simulación o evidencia académica.
4. **Explicar:** redacta la conclusión que tú puedes defender.
5. **Documentar:** si la IA influyó en un producto evaluado, registra la interacción en la bitácora de TermoIA-UAMC.

**Una segunda IA no cuenta, por sí sola, como verificación independiente.**


## 0. Diagnóstico: ¿qué piensas ahora?
Responde **verdadero o falso** y justifica en una línea. Al final del cuaderno regresarás a estas respuestas; no las borres.

1. Un cuerpo caliente contiene más calor que uno frío.
2. Si la temperatura de un gas no cambia, el gas no intercambia calor con su entorno.
3. En un proceso adiabático la temperatura no cambia.
4. Como $\Delta U$ de un ciclo es cero, el calor neto y el trabajo neto del ciclo también son cero.
5. El área bajo cualquier curva en el diagrama $P$–$V$ es el trabajo, aunque el proceso sea irreversible.
6. En bioquímica casi siempre se puede usar $\Delta H \approx \Delta U$.

*Mis respuestas iniciales:*

1.
2.
3.
4.
5.
6.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

R = 8.314462618        # J/(mol·K) = kPa·L/(mol·K)
plt.rcParams.update({"figure.figsize": (6.5, 4.2), "axes.grid": True, "grid.alpha": 0.3})

def integra(y, x):
    """Regla del trapecio (funciona igual en versiones viejas y nuevas de NumPy)."""
    y, x = np.asarray(y, float), np.asarray(x, float)
    return float(np.sum(0.5*(y[1:] + y[:-1])*np.diff(x)))

def revela(etiqueta, valor, prediccion, unidades=""):
    """Muestra el resultado solo si ya registraste una predicción numérica."""
    if prediccion is None:
        print(f"[{etiqueta}] Registra primero tu predicción. Resultado oculto.")
        return False
    print(f"[{etiqueta}] tu predicción: {prediccion:.4g} {unidades}   |   cálculo: {valor:.4g} {unidades}")
    if valor != 0:
        print(f"    diferencia relativa: {100*(prediccion - valor)/abs(valor):+.1f} %")
    return True

def revela_orden(etiqueta, valores, prediccion, tol=1e-6):
    """valores: dict nombre -> número. prediccion: lista de nombres de mayor a menor."""
    if prediccion is None:
        print(f"[{etiqueta}] Registra primero tu predicción. Resultado oculto.")
        return False
    if sorted(prediccion) != sorted(valores):
        print(f"[{etiqueta}] Usa exactamente estos nombres: {list(valores)}")
        return False
    print(f"[{etiqueta}] tu orden:")
    for nombre in prediccion:
        print(f"    {nombre:12s} {valores[nombre]:10.1f}")
    ok = all(valores[a] >= valores[b] - tol for a, b in zip(prediccion, prediccion[1:]))
    print("    ✔ tu orden es consistente con el cálculo" if ok else "    ✘ tu orden no coincide: identifica dónde y por qué")
    return True

def verifica(condicion, mensaje):
    print(("✔ " if condicion else "✘ ") + mensaje)

## 1. Calentamiento a volumen constante
A volumen constante $w=0$, así que $\Delta U = q_V = n\,C_{V,m}\,\Delta T$. Para un gas ideal monoatómico, $C_{V,m} = \tfrac{3}{2}R$.

**Predicción 1.1.** Si duplicas la cantidad de sustancia (mismo $\Delta T$), ¿por qué factor cambia $\Delta U$?

**Predicción 1.2.** Si el gas es diatómico ($C_{V,m}\approx\tfrac{5}{2}R$ a temperatura ambiente) en vez de monoatómico, ¿por qué factor cambia $\Delta U$?

In [ ]:
pred_factor_n = None            # escribe un número ANTES de ejecutar la celda siguiente
pred_factor_diatomico = None

In [ ]:
def dU_isocoro(n, Cv_m, T1, T2):
    return n*Cv_m*(T2 - T1)

T1, T2 = 298.15, 350.0
base = dU_isocoro(1.0, 1.5*R, T1, T2)
print(f"ΔU (1 mol, monoatómico) = {base:.1f} J")
revela("1.1", dU_isocoro(2.0, 1.5*R, T1, T2)/base, pred_factor_n)
revela("1.2", dU_isocoro(1.0, 2.5*R, T1, T2)/base, pred_factor_diatomico)

**Contraste 1.** ¿Qué propiedad de $U$ ilustra la predicción 1.1? ¿De dónde sale la diferencia entre gas monoatómico y diatómico (piensa en las formas en que una molécula puede almacenar energía)? ¿Esperarías que $C_{V,m}$ del N₂ siga siendo $\tfrac52 R$ a 2000 K?

*Tu respuesta:*

## 2. Mismo $\Delta T$, distinta trayectoria: volumen constante vs. presión constante
Calentamos 1 mol de gas ideal monoatómico de 298.15 K a 350 K por dos caminos: **(a)** a volumen constante y **(b)** a presión constante, $P = 100$ kPa.

**Predicción 2.1.** ¿Cuál requiere más calor? Estima el cociente $q_P/q_V$.

**Predicción 2.2.** ¿$\Delta U$ en (b) es mayor, menor o igual que en (a)? Escribe el cociente $\Delta U_b/\Delta U_a$.

In [ ]:
pred_cociente_q = None
pred_cociente_dU = None

In [ ]:
n, Cv_m = 1.0, 1.5*R
Cp_m = Cv_m + R                     # gas ideal
P = 100.0                           # kPa
dT = T2 - T1

# (a) volumen constante
w_a = 0.0
q_a = n*Cv_m*dT
dU_a = q_a + w_a

# (b) presión constante
V1, V2 = n*R*T1/P, n*R*T2/P         # L
w_b = -P*(V2 - V1)                  # J, porque kPa·L = J
q_b = n*Cp_m*dT
dU_b = q_b + w_b

if revela("2.1", q_b/q_a, pred_cociente_q) and revela("2.2", dU_b/dU_a, pred_cociente_dU):
    print(f"\n(a) q = {q_a:8.1f} J   w = {w_a:8.1f} J   ΔU = {dU_a:8.1f} J")
    print(f"(b) q = {q_b:8.1f} J   w = {w_b:8.1f} J   ΔU = {dU_b:8.1f} J")

**Contraste 2.** ¿A dónde va el calor "extra" del camino (b)? ¿Qué te dice la igualdad de $\Delta U$ en ambos caminos sobre la dependencia de $U$ con $T$, $P$ y $V$ en un gas ideal?

*Tu respuesta:*

### Tu turno 2.3 — entalpía
Define $H = U + PV$. Calcula $\Delta H$ para el camino (b) y comprueba que (i) $\Delta H = q_P$ y (ii) $\Delta H - \Delta U = nR\Delta T$.

In [ ]:
dH_b = None    # TODO: ΔH = ΔU + Δ(PV). Usa P, V1, V2 y dU_b.

if dH_b is not None:
    verifica(abs(dH_b - q_b) < 1e-6, "ΔH = q_P a presión constante")
    verifica(abs((dH_b - dU_b) - n*R*dT) < 1e-6, "ΔH − ΔU = nRΔT para un gas ideal")

### Tu turno 2.4 — ¿y en una disolución acuosa?
Calienta 1.000 kg de agua líquida de 25 °C a 35 °C a 100 kPa. Datos: $c = 4.18$ J/(g·K); $V \approx 1.003$ L; coeficiente de expansión térmica volumétrica promedio en ese intervalo $\beta \approx 3.0\times10^{-4}$ K⁻¹, con $\Delta V \approx \beta V \Delta T$.

Calcula $q_P$, el trabajo de expansión y el cociente $|w|/q_P$. Antes de calcular, escribe tu estimación del orden de magnitud de ese cociente.

In [ ]:
pred_orden_cociente = None      # p. ej. 1e-2

m, c, beta, V = 1000.0, 4.18, 3.0e-4, 1.003    # g, J/(g·K), 1/K, L
dT_agua, P = 10.0, 100.0                       # K, kPa

q_P_agua = None    # TODO (J)
dV = None          # TODO (L)
w_agua = None      # TODO (J), convención: trabajo sobre el sistema

if None not in (q_P_agua, dV, w_agua):
    revela("2.4", abs(w_agua)/q_P_agua, pred_orden_cociente)

**Contraste 2.4.** Con este resultado, revisa tu respuesta al enunciado 6 del diagnóstico. ¿En qué situaciones de interés biológico **no** podrías usar $\Delta H\approx\Delta U$? (Pista: ¿qué tiene de especial el gas ideal del inciso 2.3?)

*Tu respuesta:*

## 3. Diagramas $P$–$V$: cuatro procesos desde el mismo estado
Estado inicial A: 1 mol de gas ideal monoatómico, $T = 298.15$ K, $P = 100$ kPa ($V\approx 24.8$ L). Desde A:

- **isobárico**: expansión a presión constante hasta duplicar el volumen;
- **isotérmico** reversible: expansión hasta duplicar el volumen;
- **adiabático** reversible: expansión hasta duplicar el volumen ($PV^\gamma=$ cte, con $\gamma = C_P/C_V = 5/3$);
- **isocórico**: calentamiento a volumen constante hasta duplicar la presión.

**Predicción 3.1.** Ordena de mayor a menor la **presión final** de las tres expansiones.

**Predicción 3.2.** Ordena de mayor a menor el **trabajo hecho por el gas** ($-w$) en las tres expansiones.

In [ ]:
# Usa exactamente estos nombres: "isobarico", "isotermico", "adiabatico"
pred_orden_Pfinal = None       # p. ej. ["isotermico", "isobarico", "adiabatico"]
pred_orden_trabajo = None

In [ ]:
# --- Generadores de trayectorias cuasiestáticas: devuelven arreglos (V en L, P en kPa) ---
def isobara(P, Vi, Vf, N=200):
    V = np.linspace(Vi, Vf, N)
    return V, np.full(N, float(P))

def isocora(V, Pi, Pf, N=200):
    return np.full(N, float(V)), np.linspace(Pi, Pf, N)

def isoterma(n, T, Vi, Vf, N=400):
    V = np.linspace(Vi, Vf, N)
    return V, n*R*T/V

def adiabata(Pi, Vi, Vf, gamma, N=400):
    V = np.linspace(Vi, Vf, N)
    return V, Pi*(Vi/V)**gamma

def flechas(ax, V, P, color, k=2):
    """Dibuja k flechas sobre una trayectoria para indicar el sentido del proceso."""
    for frac in (np.arange(k) + 0.5)/k:
        i = int(frac*(len(V) - 2))
        ax.annotate("", xy=(V[i+1], P[i+1]), xytext=(V[i], P[i]),
                    arrowprops=dict(arrowstyle="-|>", color=color, lw=1.5, mutation_scale=15))

In [ ]:
n, T_A, P_A, gamma = 1.0, 298.15, 100.0, 5/3
V_A = n*R*T_A/P_A
V_B = 2*V_A

procesos = {
    "isobarico":  isobara(P_A, V_A, V_B),
    "isotermico": isoterma(n, T_A, V_A, V_B),
    "adiabatico": adiabata(P_A, V_A, V_B, gamma),
    "isocorico":  isocora(V_A, P_A, 2*P_A),
}
colores = dict(isobarico="C0", isotermico="C1", adiabatico="C2", isocorico="C3")

if pred_orden_Pfinal is None:
    print("Registra pred_orden_Pfinal para ver el diagrama.")
else:
    fig, ax = plt.subplots()
    for nombre, (V, P) in procesos.items():
        ax.plot(V, P, color=colores[nombre], label=nombre)
        flechas(ax, V, P, colores[nombre])
    ax.plot(V_A, P_A, "ko")
    ax.annotate("A", (V_A, P_A), xytext=(-14, -14), textcoords="offset points")
    ax.set_xlabel("V (L)"); ax.set_ylabel("P (kPa)"); ax.set_title("Cuatro procesos desde A")
    ax.legend(); plt.show()

In [ ]:
# Trabajo hecho SOBRE el gas: w = −∫P dV, por integración numérica de cada trayectoria
w = {nombre: -integra(P, V) for nombre, (V, P) in procesos.items()}

# Valores analíticos para comparar
Cv_m = 1.5*R
T_ad = T_A*(V_A/V_B)**(gamma - 1)
w_analitico = {
    "isobarico":  -P_A*(V_B - V_A),
    "isotermico": -n*R*T_A*np.log(V_B/V_A),
    "adiabatico":  n*Cv_m*(T_ad - T_A),      # q = 0  ⇒  w = ΔU
    "isocorico":   0.0,
}
expansiones = ["isobarico", "isotermico", "adiabatico"]
a = revela_orden("3.1 presión final", {k: procesos[k][1][-1] for k in expansiones}, pred_orden_Pfinal)
b = revela_orden("3.2 trabajo del gas", {k: -w[k] for k in expansiones}, pred_orden_trabajo)
if a and b:
    print(f"\n{'proceso':12s} {'w numérico (J)':>15s} {'w analítico (J)':>16s} {'P final (kPa)':>14s}")
    for k in procesos:
        print(f"{k:12s} {w[k]:15.1f} {w_analitico[k]:16.1f} {procesos[k][1][-1]:14.1f}")

In [ ]:
# El trabajo como área bajo la curva
fig, axs = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, nombre in zip(axs, ["isotermico", "adiabatico"]):
    V, P = procesos[nombre]
    ax.plot(V, P, color=colores[nombre])
    ax.fill_between(V, 0, P, color=colores[nombre], alpha=0.25)
    ax.set_title(f"{nombre}: −w = área = {-w[nombre]:.0f} J")
    ax.set_xlabel("V (L)")
axs[0].set_ylabel("P (kPa)")
plt.show()

**Contraste 3.** ¿Por qué la adiabática cae más rápido que la isoterma? Explícalo con la primera ley: si $q=0$, ¿de dónde sale la energía para el trabajo y qué le pasa a $T$? ¿Por qué el trabajo del proceso isocórico es cero aunque la presión se duplique? Revisa tus respuestas a los enunciados 2 y 3 del diagnóstico.

*Tu respuesta:*

### Tu turno 3.3 — las pendientes
Comprueba numéricamente que, en el punto A, $(dP/dV)_\text{adiabática} / (dP/dV)_\text{isoterma} = \gamma$. Usa `np.gradient(P, V, edge_order=2)`; el primer elemento del arreglo corresponde a A. Después, demuestra el resultado analíticamente en una línea.

In [ ]:
pend_iso = None    # TODO
pend_ad = None     # TODO
if None not in (pend_iso, pend_ad):
    print(f"cociente de pendientes = {pend_ad/pend_iso:.4f}    γ = {gamma:.4f}")

### Tu turno 3.4 — la adiabática cumple $\Delta U = w$
Calcula la temperatura al inicio y al final de la trayectoria adiabática a partir de $P$ y $V$ (gas ideal) y comprueba que $n C_{V,m}\Delta T$ coincide con el trabajo obtenido por integración numérica.

In [ ]:
V, P = procesos["adiabatico"]
T_i = None    # TODO
T_f = None    # TODO
if None not in (T_i, T_f):
    dU_ad = n*Cv_m*(T_f - T_i)
    verifica(abs(dU_ad - w["adiabatico"]) < 0.5,
             f"en la adiabática ΔU = w ({dU_ad:.1f} J vs {w['adiabatico']:.1f} J), es decir, q = 0")

### ★ Ampliación 3.5 — ¿cuánto confiar en la integral numérica?
Calcula el error absoluto de `integra` para la isoterma en función del número de puntos $N$ y grafícalo en escala log–log. ¿Qué pendiente obtienes y qué significa? ¿Cuántos puntos necesitas para un error menor que 0.1 J?

In [ ]:
Ns = [5, 10, 20, 50, 100, 200, 500, 1000]
errores = []
for N in Ns:
    V, P = isoterma(n, T_A, V_A, V_B, N=N)
    # TODO: agrega a 'errores' el valor absoluto de (−integra(P, V) − w_analitico["isotermico"])
    pass

if len(errores) == len(Ns):
    pendiente = np.polyfit(np.log(Ns), np.log(errores), 1)[0]
    plt.loglog(Ns, errores, "o-")
    plt.xlabel("N (puntos)"); plt.ylabel("|error| (J)"); plt.title(f"pendiente ≈ {pendiente:.2f}")
    plt.show()

## 4. El trabajo depende de la trayectoria; $\Delta U$ no
Mismos estados inicial y final para todos los caminos: **A** (200 kPa, ≈12.4 L) y **B** (100 kPa, ≈24.8 L), ambos a 298.15 K; 1 mol de gas ideal monoatómico.

1. `reversible`: expansión isotérmica reversible.
2. `via_C`: expansión a 200 kPa hasta $V_B$ (A→C) y luego enfriamiento a volumen constante hasta 100 kPa (C→B).
3. `via_D`: enfriamiento a volumen constante hasta 100 kPa (A→D) y luego expansión a 100 kPa (D→B).
4. `1_paso`: se retira de golpe parte de la carga del pistón; el gas se expande rápido contra $P_\text{ext}=100$ kPa constante, en contacto con un baño a 298.15 K, y termina en B.
5. `4_pasos`: igual que el anterior, pero la carga se retira en 4 escalones.

**Predicción 4.1.** Ordena los cinco caminos de mayor a menor trabajo hecho por el gas ($-w$).

**Predicción 4.2.** ¿Cuánto vale $\Delta U$ en cada camino? ¿Y $q$?

In [ ]:
# Nombres: "reversible", "via_C", "via_D", "1_paso", "4_pasos"
pred_orden_caminos = None
pred_dU_caminos = None      # un solo número (J) si crees que es igual en todos; si no, déjalo en None y explícalo abajo

In [ ]:
n, T = 1.0, 298.15
P_A, P_B = 200.0, 100.0
V_A, V_B = n*R*T/P_A, n*R*T/P_B
Cv_m = 1.5*R

def expansion_escalonada(k, Vi=V_A, Vf=V_B):
    """k escalones de presión externa. En cada escalón, P_ext es la presión de equilibrio
    del gas al final de ese escalón y el gas se expande irreversiblemente contra ella.
    Devuelve w (sobre el gas) y los arreglos (V, P_ext) para dibujar la escalera."""
    Vs = np.linspace(Vi, Vf, k + 1)
    Pext = n*R*T/Vs[1:]
    w = -np.sum(Pext*np.diff(Vs))
    return w, np.repeat(Vs, 2)[1:-1], np.repeat(Pext, 2)

caminos = {}
caminos["reversible"] = -n*R*T*np.log(V_B/V_A)
caminos["via_C"] = -P_A*(V_B - V_A)
caminos["via_D"] = -P_B*(V_B - V_A)
caminos["1_paso"], V1p, P1p = expansion_escalonada(1)
caminos["4_pasos"], V4p, P4p = expansion_escalonada(4)

dU_AB = n*Cv_m*(T - T)      # U de un gas ideal solo depende de T, y T_A = T_B
if revela_orden("4.1", {k: -v for k, v in caminos.items()}, pred_orden_caminos):
    print(f"\n{'camino':12s} {'w (J)':>9s} {'q (J)':>9s} {'ΔU (J)':>8s}")
    for k, wk in caminos.items():
        print(f"{k:12s} {wk:9.1f} {dU_AB - wk:9.1f} {dU_AB:8.1f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
V, P = isoterma(n, T, V_A, V_B)
ax.plot(V, P, "C1", lw=2, label="1: isoterma reversible")
ax.plot([V_A, V_B, V_B], [P_A, P_A, P_B], "C0", label="2: vía C")
ax.plot([V_A, V_A, V_B], [P_A, P_B, P_B], "C2", lw=5, alpha=0.35, label="3: vía D")
ax.plot(V1p, P1p, "k:", lw=2, label="4: $P_{ext}$, 1 paso (coincide con el tramo D→B)")
ax.plot(V4p, P4p, "C3--", label="5: $P_{ext}$, 4 pasos")
ax.plot([V_A, V_B], [P_A, P_B], "ko")
ax.annotate("A", (V_A, P_A), xytext=(-14, 4), textcoords="offset points")
ax.annotate("B", (V_B, P_B), xytext=(6, 4), textcoords="offset points")
ax.set_xlabel("V (L)"); ax.set_ylabel("P (kPa)"); ax.legend(fontsize=8)
plt.show()

⚠ **Cuidado con las líneas punteadas.** En los caminos 4 y 5 lo que se dibuja es $P_\text{ext}$, no la presión del gas. Durante una expansión rápida el gas no tiene una presión ni una temperatura únicas (hay gradientes y turbulencia), así que el proceso **no** es una sucesión de estados de equilibrio y no puede dibujarse como una curva en el diagrama del gas. El área bajo $P_\text{ext}$ sí da el trabajo, $w = -\int P_\text{ext}\,dV$. Revisa tu respuesta al enunciado 5 del diagnóstico.

**Contraste 4.** ¿Por qué `via_D` y `1_paso` dan el mismo trabajo si son procesos físicamente distintos? ¿Por qué $q$ es diferente en cada camino si $\Delta U$ es el mismo? Escribe con tus palabras la diferencia entre una función de estado y una cantidad que depende de la trayectoria.

*Tu respuesta:*

### Tu turno 4.3 — el reversible como límite
Calcula $w_k/w_\text{rev}$ para $k = 1,\dots,200$ escalones y grafícalo. ¿Cuál es el menor $k$ que alcanza el 99 % del trabajo reversible? ¿Qué dice esto sobre lo que significa "reversible" en la práctica?

In [ ]:
ks = np.arange(1, 201)
cocientes = None     # TODO: arreglo con w_k / w_rev para cada k (usa expansion_escalonada)
k99 = None           # TODO: menor k con cociente ≥ 0.99

if cocientes is not None:
    plt.plot(ks, cocientes)
    plt.axhline(1, color="k", lw=0.8); plt.axhline(0.99, ls=":", color="k")
    plt.xscale("log"); plt.xlabel("número de escalones k"); plt.ylabel("$w_k / w_{rev}$")
    plt.show()
    print("k mínimo para alcanzar 99 %:", k99)

### Tu turno 4.4 — compresión irreversible
Ahora regresa de B a A comprimiendo en $k$ escalones (en cada escalón, $P_\text{ext}$ es la presión de equilibrio al final del escalón, que ahora es mayor). Antes de programar:

- ¿La compresión en 1 paso requiere **más o menos** trabajo que la compresión reversible?
- En el ciclo A→B→A con ambos tramos en 1 paso, ¿qué signo tiene el trabajo neto **sobre el gas**? ¿Qué le pasó a esa energía?

Pista: revisa si `expansion_escalonada` ya sirve para comprimir.

In [ ]:
pred_signo_ciclo = None      # "+" o "-"

def compresion_escalonada(k, Vi=V_B, Vf=V_A):
    # TODO: devuelve el trabajo sobre el gas al comprimir de Vi a Vf en k escalones
    return None

if compresion_escalonada(1) is not None:
    w_rev_comp = n*R*T*np.log(V_B/V_A)
    for k in [1, 4, 100]:
        print(f"k = {k:3d}: w compresión = {compresion_escalonada(k):8.1f} J   (reversible: {w_rev_comp:.1f} J)")
    w_ciclo = caminos["1_paso"] + compresion_escalonada(1)
    if pred_signo_ciclo is None:
        print("Registra pred_signo_ciclo para ver el trabajo neto del ciclo.")
    else:
        print(f"w neto sobre el gas en A→B→A (1 paso + 1 paso) = {w_ciclo:.1f} J   (tu predicción: {pred_signo_ciclo})")

**Contraste 4.4.** El gas regresó a su estado inicial ($\Delta U = 0$), pero el entorno no. Explica qué cambió en el entorno. ¿Hay algún número de escalones para el cual el ciclo irreversible no deje rastro?

*Tu respuesta:*

## 5. Ciclos
En un ciclo el estado final es el inicial, así que $\Delta U_\text{ciclo}=0$ porque $U$ es función de estado. En cambio, $q_\text{ciclo}$ y $w_\text{ciclo}$ no tienen por qué anularse: el trabajo neto es el **área encerrada** y su signo depende del sentido en que se recorre el ciclo.

La función `analiza_ciclo` recibe una lista de tramos, integra numéricamente $w$ en cada uno, calcula $\Delta U$ a partir de las temperaturas de los extremos y obtiene $q = \Delta U - w$. Sirve para **cualquier** trayectoria cuasiestática de un gas ideal, no solo para las de libro.

In [ ]:
def analiza_ciclo(tramos, n=1.0, Cv_m=1.5*R, titulo="", mostrar=True):
    """tramos: lista de (nombre, (V, P)) consecutivos. V en L, P en kPa.
    Devuelve una lista de filas con T_i, T_f, w, q, ΔU por tramo y un diccionario de totales."""
    filas = []
    for nombre, (V, P) in tramos:
        w = -integra(P, V)
        Ti, Tf = P[0]*V[0]/(n*R), P[-1]*V[-1]/(n*R)
        dU = n*Cv_m*(Tf - Ti)
        filas.append(dict(tramo=nombre, Ti=Ti, Tf=Tf, w=w, q=dU - w, dU=dU))
    for (n1, (Va, Pa)), (n2, (Vb, Pb)) in zip(tramos, tramos[1:] + tramos[:1]):
        if not (np.isclose(Va[-1], Vb[0], rtol=1e-6) and np.isclose(Pa[-1], Pb[0], rtol=1e-6)):
            print(f"⚠ El tramo '{n1}' no termina donde empieza '{n2}': el ciclo no está cerrado.")
    tot = {k: sum(f[k] for f in filas) for k in ("w", "q", "dU")}
    if mostrar:
        print(titulo)
        print(f"{'tramo':16s} {'T_i (K)':>8s} {'T_f (K)':>8s} {'w (J)':>9s} {'q (J)':>9s} {'ΔU (J)':>9s}")
        for f in filas:
            print(f"{f['tramo']:16s} {f['Ti']:8.1f} {f['Tf']:8.1f} {f['w']:9.1f} {f['q']:9.1f} {f['dU']:9.1f}")
        print(f"{'TOTAL':16s} {'':8s} {'':8s} {tot['w']:9.1f} {tot['q']:9.1f} {tot['dU']:9.1f}")
    return filas, tot

def dibuja_ciclo(tramos, titulo=""):
    fig, ax = plt.subplots()
    Vc = np.concatenate([V for _, (V, P) in tramos])
    Pc = np.concatenate([P for _, (V, P) in tramos])
    ax.fill(Vc, Pc, alpha=0.15, color="gray")
    for i, (nombre, (V, P)) in enumerate(tramos):
        ax.plot(V, P, color=f"C{i}", label=nombre)
        flechas(ax, V, P, f"C{i}", k=1)
    ax.set_xlabel("V (L)"); ax.set_ylabel("P (kPa)"); ax.set_title(titulo); ax.legend(fontsize=8)
    plt.show()

### 5.1 Ciclo rectangular
1 mol de gas monoatómico: A (10 L, 200 kPa) → B (30 L, 200 kPa) → C (30 L, 100 kPa) → D (10 L, 100 kPa) → A.

**Predicción 5.1.** Dibuja el ciclo en papel. ¿Cuánto vale el trabajo neto **sobre el gas** (con signo)? ¿Qué signo tiene el calor neto? ¿En qué vértice es máxima la temperatura?

In [ ]:
pred_w_neto_rect = None     # J, con signo

In [ ]:
rect = [
    ("A→B isobara", isobara(200, 10, 30)),
    ("B→C isocora", isocora(30, 200, 100)),
    ("C→D isobara", isobara(100, 30, 10)),
    ("D→A isocora", isocora(10, 100, 200)),
]
filas, tot = analiza_ciclo(rect, mostrar=False)
if revela("5.1", tot["w"], pred_w_neto_rect, "J"):
    analiza_ciclo(rect, titulo="\nCiclo rectangular")
    verifica(abs(tot["dU"]) < 1e-6, "ΔU del ciclo = 0")
    dibuja_ciclo(rect, "Ciclo rectangular")

**Contraste 5.1.** ¿En qué tramos entra calor al gas y en cuáles sale? ¿Por qué $\Delta U_\text{ciclo}=0$ sirve como prueba de que el **código** está bien, más que como resultado físico? ¿Por qué $q_\text{ciclo}\neq 0$? Revisa el enunciado 4 del diagnóstico.

*Tu respuesta:*

### Tu turno 5.2 — invierte el sentido
Construye `rect_inverso`, que recorre el mismo rectángulo en sentido contrario (A→D→C→B→A). Predice el signo de $w$ y $q$ netos. ¿Qué tipo de dispositivo representa cada sentido?

Pista: `V[::-1]` invierte un arreglo y `reversed(lista)` invierte el orden de los tramos.

In [ ]:
pred_w_neto_inverso = None
rect_inverso = None        # TODO

if rect_inverso is not None:
    filas, tot = analiza_ciclo(rect_inverso, mostrar=False)
    if revela("5.2", tot["w"], pred_w_neto_inverso, "J"):
        analiza_ciclo(rect_inverso, titulo="\nCiclo rectangular invertido")

### Tu turno 5.3 — un ciclo con isoterma
1 mol de gas monoatómico. A: $T_h = 400$ K, $V = 10$ L. A→B: expansión isotérmica reversible hasta 30 L. B→C: compresión isobárica hasta 10 L. C→A: calentamiento isocórico hasta regresar a A.

Construye el ciclo, analízalo y calcula el calor absorbido $q_\text{in}$ (suma de los $q>0$) y el rendimiento $\eta = -w_\text{neto}/q_\text{in}$. Después cambia $V_2$ a 15 L y a 60 L: ¿cómo cambia $\eta$? Anota lo que observas; regresaremos a esto con la segunda ley.

In [ ]:
T_h, V_1, V_2 = 400.0, 10.0, 30.0
P_1 = None          # TODO: presión en A (kPa)
P_2 = None          # TODO: presión en B (kPa)
ciclo_iso = None    # TODO: lista de 3 tramos (isoterma, isobara, isocora)

if ciclo_iso is not None:
    filas, tot = analiza_ciclo(ciclo_iso, titulo="Isoterma + isobara + isocora")
    verifica(abs(tot["dU"]) < 1e-6, "ΔU del ciclo = 0")
    verifica(abs(filas[0]["dU"]) < 1e-6 and abs(filas[0]["q"] + filas[0]["w"]) < 1e-6,
             "en la isoterma ΔU = 0 y q = −w")
    q_in = sum(f["q"] for f in filas if f["q"] > 0)
    print(f"q absorbido = {q_in:.1f} J   rendimiento η = {-tot['w']/q_in:.3f}")
    dibuja_ciclo(ciclo_iso, "Ciclo con isoterma")

### Tu turno 5.4 — un ciclo que no viene en los libros
El ciclo es una elipse en el plano $P$–$V$: $V = 25 + 10\cos\theta$ (L), $P = 150 + 50\sin\theta$ (kPa), con $\theta$ de 0 a $2\pi$.

**Predicción 5.4.** Usa solo geometría: ¿cuánto vale $w_\text{neto}$ sobre el gas, con signo? (Determina el sentido de recorrido evaluando dos o tres valores de $\theta$.) Luego modifica la parametrización para que el ciclo funcione como **motor**.

In [ ]:
pred_w_elipse = None
theta = np.linspace(0, 2*np.pi, 2001)
V_e = 25 + 10*np.cos(theta)
P_e = 150 + 50*np.sin(theta)

elipse = [("elipse", (V_e, P_e))]
filas, tot = analiza_ciclo(elipse, mostrar=False)
if revela("5.4", tot["w"], pred_w_elipse, "J"):
    dibuja_ciclo(elipse, "Ciclo elíptico")

## 6. Cacería de errores
Cada celda corre sin errores de Python, pero tiene errores de **física**. Para cada una:

1. Antes de tocar el código, estima el orden de magnitud correcto del resultado.
2. Lista los errores que encuentres (puede haber más de uno, o ninguno en alguna línea sospechosa).
3. Corrige en una celda **nueva** debajo; no borres la versión con errores.
4. **Auditoría con IA (PIVED):** sólo después de tu diagnóstico, pide a una IA que revise **una** de las cuatro celdas. Registra qué error detectó, cuál omitió y si introdujo alguna afirmación nueva. Verifica cada punto antes de aceptarlo.

In [ ]:
# 6.1  Trabajo hecho POR el gas en la expansión isotérmica reversible
#      de 2 mol de N2 desde 5 L hasta 15 L a 25 °C.
n = 2
T = 25                     # temperatura
V1, V2 = 5, 15             # L
R_ = 0.082                 # constante de los gases
w_por_gas = n*R_*T*np.log(V2/V1)
print(f"Trabajo hecho por el gas = {w_por_gas:.2f} J")

In [ ]:
# 6.2  Calor necesario para llevar 250 mL de medio de cultivo (≈ agua) de 4 °C a 37 °C.
m = 0.250                  # kg
c = 4.18                   # J/(g·K)
dT = 37 - 4                # °C
q = m*c*dT
print(f"q = {q:.1f} J")

In [ ]:
# 6.3  Un gas recibe 500 J de calor y se expande haciendo 200 J de trabajo sobre el entorno.
q = 500.0
w = 200.0                  # "trabajo", tomado de un libro de física
dU = q + w
print(f"ΔU = {dU:.0f} J")

In [ ]:
# 6.4  Expansión adiabática reversible de aire (γ = 1.4) de 1 L a 2 L; T inicial 300 K.
gamma = 1.4
T1 = 300.0
V1, V2 = 1.0, 2.0
T2 = T1*(V2/V1)**(gamma - 1)
print(f"T final = {T2:.1f} K")

**Reflexión 6.** Para cada error, ¿qué verificación rápida lo habría detectado (unidades, signo, orden de magnitud, caso límite, sentido físico)? En la celda 6.2, ¿es un error usar °C para $\Delta T$? ¿Y en la 6.1 para $T$? Escribe una **lista de control** de 4–5 puntos que usarás en el resto del curso.

*Mi lista de control:*

1.
2.
3.
4.

## 7. Calorimetría
Un calorímetro intenta aislar el sistema para que el balance de energía sea simplemente $\sum_i q_i = 0$: lo que unos cuerpos pierden, otros lo ganan. En la práctica el aislamiento es imperfecto y el propio calorímetro absorbe calor. Esta sección va de lo ideal a lo realista.

In [ ]:
c_agua = 4.184     # J/(g·K)
c_hielo = 2.09     # J/(g·K)
L_fus = 334.0      # J/g, entalpía de fusión del hielo a 0 °C

### 7.1 Mezcla de agua en un calorímetro
Un calorímetro con capacidad calorífica $C_\text{cal} = 50$ J/K contiene 150 g de agua; ambos están a 20.0 °C. Se añaden 200 g de agua a 80.0 °C.

**Tu turno.** Completa `T_final_mezcla`, que recibe una lista de cuerpos $(C_i, T_i)$, con $C_i$ la capacidad calorífica total de cada uno en J/K, y devuelve la temperatura de equilibrio a partir de $\sum_i C_i\,(T_f - T_i) = 0$.

**Predicción 7.1.** Sin calcular con precisión: ¿$T_f$ queda por arriba o por debajo del promedio ponderado por masa del agua, 54.3 °C? Escribe un valor.

In [ ]:
def T_final_mezcla(cuerpos):
    """cuerpos: lista de (C_i en J/K, T_i en °C). Devuelve T_f en °C."""
    # TODO
    return None

pred_Tf_mezcla = None

In [ ]:
cuerpos = [(200*c_agua, 80.0), (150*c_agua, 20.0), (50.0, 20.0)]
Tf = T_final_mezcla(cuerpos)
if Tf is not None and revela("7.1", Tf, pred_Tf_mezcla, "°C"):
    print(f"Ignorando el calorímetro: {T_final_mezcla(cuerpos[:2]):.2f} °C")
    verifica(abs(sum(C*(Tf - Ti) for C, Ti in cuerpos)) < 1e-9, "Σ q_i = 0")

**Contraste 7.1.** ¿Qué error relativo en $\Delta T$ del agua caliente introduce ignorar el calorímetro? ¿Por qué aquí sí se pueden usar temperaturas en °C, a diferencia de la celda 6.1?

*Tu respuesta:*

### 7.2 Calibrar el calorímetro: réplicas e incertidumbre
Para conocer $C_\text{cal}$ se consideran 5 **ensayos didácticos simulados**: el calorímetro con $m_c$ g de agua fría a $T_c$ recibe $m_h$ g de agua caliente a $T_h$, y se registra $T_f$. Para el ejercicio, modelaremos el error de lectura del termómetro como ruido normal con desviación estándar $\sigma_T=0.05$ K.

**Tu turno.**
1. Despeja $C_\text{cal}$ del balance $m_h c\,(T_f - T_h) + (m_c c + C_\text{cal})(T_f - T_c) = 0$ y calcúlalo para cada ensayo. Escribe la función usando solo operaciones de NumPy para que funcione también con arreglos.
2. ¿Hay algún ensayo sospechoso? Propón una causa **física** plausible (no basta con "se ve raro"). **No lo elimines sólo por ser discrepante**: indica qué observación o repetición adicional necesitarías para justificar excluirlo.
3. Con Monte Carlo, perturba $T_h$, $T_c$ y $T_f$ del ensayo 1 con ruido normal de σ = 0.05 K y obtén la desviación estándar de $C_\text{cal}$. Compárala con la dispersión entre ensayos.

In [ ]:
#            m_h (g), T_h (°C), m_c (g), T_c (°C), T_f (°C)
ensayos = np.array([
    ( 105.3,  61.5,   96.4,  21.1,  41.16),
    (  99.0,  61.1,   81.3,  20.1,  41.33),
    ( 103.9,  60.4,  101.3,  18.5,  38.59),
    ( 101.9,  60.3,  100.4,  20.2,  38.80),
    ( 100.2,  60.5,  100.8,  20.9,  39.61)
])

def C_cal_de_ensayo(m_h, T_h, m_c, T_c, T_f):
    # TODO
    return None

if C_cal_de_ensayo(*ensayos[0]) is not None:
    C_ens = np.array([C_cal_de_ensayo(*fila) for fila in ensayos])
    for i, Ci in enumerate(C_ens, 1):
        print(f"ensayo {i}: C_cal = {Ci:6.1f} J/K")
    plt.plot(range(1, 6), C_ens, "o"); plt.xlabel("ensayo"); plt.ylabel("$C_{cal}$ (J/K)"); plt.show()

In [ ]:
rng = np.random.default_rng(0)
sigma_T, N_mc = 0.05, 10_000
m_h, T_h, m_c, T_c, T_f = ensayos[0]

C_mc = None     # TODO: arreglo de N_mc valores de C_cal con T_h, T_c y T_f perturbadas

if C_mc is not None:
    print(f"C_cal (ensayo 1) = {np.mean(C_mc):.1f} ± {np.std(C_mc):.1f} J/K  (Monte Carlo)")
    print(f"incertidumbre relativa: C_cal {100*np.std(C_mc)/np.mean(C_mc):.1f} %  vs  T_f − T_c {100*sigma_T*np.sqrt(2)/(T_f - T_c):.2f} %")

**Contraste 7.2.** ¿Por qué la incertidumbre relativa de $C_\text{cal}$ es mucho mayor que la de las temperaturas? (Pista: en la fórmula, $C_\text{cal}$ es la diferencia de dos números parecidos.) Propón **dos** cambios en el diseño del experimento que reduzcan esa incertidumbre y justifícalos con la fórmula o con una simulación.

*Tu respuesta:*

### 7.3 Hielo en agua: cuando el modelo ingenuo produce un absurdo
Se agregan $m_h$ g de hielo a $-10$ °C a 200 g de agua a 25 °C, en un calorímetro de capacidad despreciable.

La función `T_final_ingenua` supone que **todo** el hielo se funde: calienta el hielo hasta 0 °C, lo funde y luego calienta el agua resultante hasta $T_f$.

**Predicción 7.3.** Calcula a mano (orden de magnitud) cuánta energía puede ceder el agua al enfriarse hasta 0 °C y cuánta necesita el hielo para calentarse y fundirse por completo. Con eso, predice $T_f$ para (a) $m_h = 50$ g y (b) $m_h = 100$ g.

In [ ]:
pred_Tf_hielo_50 = None
pred_Tf_hielo_100 = None

def T_final_ingenua(m_h, T_h, m_a, T_a):
    """Supone que TODO el hielo se funde."""
    return (m_a*c_agua*T_a - m_h*c_hielo*(0 - T_h) - m_h*L_fus) / ((m_a + m_h)*c_agua)

for m_h, pred in [(50, pred_Tf_hielo_50), (100, pred_Tf_hielo_100)]:
    revela(f"7.3 ingenua, {m_h} g", T_final_ingenua(m_h, -10, 200, 25), pred, "°C")

**Contraste 7.3.** Uno de los dos resultados es físicamente imposible. ¿Cuál y por qué? ¿Qué supuesto del modelo falla?

### Tu turno 7.3b — un modelo que distingue casos
Escribe `hielo_en_agua(m_h, T_h, m_a, T_a)` que devuelva `(T_f, m_fundida)` y que distinga dos casos: **(A)** todo el hielo se funde y $T_f \geq 0$ °C; **(B)** solo se funde una parte y $T_f = 0$ °C. La función `balance_hielo` comprueba que la energía se conserve y que la respuesta tenga sentido físico.

★ Ampliación: ¿existe un tercer caso? Diseña datos en los que ocurra y extiende la función.

In [ ]:
def hielo_en_agua(m_h, T_h, m_a, T_a):
    # TODO: decide en qué caso estás comparando energías y devuelve (T_f, m_fundida)
    return None

def balance_hielo(m_h, T_h, m_a, T_a, T_f, m_f):
    """Suma de calores (debe ser ~0). Válido si T_f ≥ 0 y el hielo no fundido está a 0 °C."""
    q_agua = m_a*c_agua*(T_f - T_a)
    q_hielo = m_h*c_hielo*(0 - T_h) + m_f*L_fus + m_f*c_agua*(T_f - 0)
    return q_agua + q_hielo

for m_h in [50, 100, 150]:
    res = hielo_en_agua(m_h, -10, 200, 25)
    if res is None:
        print("Completa hielo_en_agua."); break
    T_f, m_f = res
    print(f"m_hielo = {m_h:3d} g:  T_f = {T_f:6.2f} °C, fundido = {m_f:6.1f} g")
    verifica(abs(balance_hielo(m_h, -10, 200, 25, T_f, m_f)) < 1e-6, "  energía conservada")
    verifica(0 <= m_f <= m_h and T_f >= 0 and (m_f == m_h or T_f == 0), "  resultado físicamente consistente")

### ★ Ampliación 7.4 — Un experimento realista no es adiabático: entalpía de neutralización
La siguiente **serie temporal es sintética, pero físicamente realista**. Se modela la mezcla de 50.0 mL de HCl 1.00 M con 50.0 mL de NaOH 1.00 M en un calorímetro de vaso de unicel en $t = 120$ s. La capacidad calorífica total es $C = 100\ \text{g}\times4.18\ \text{J/(g·K)} + 30\ \text{J/K} = 448$ J/K. La temperatura ambiente es 23.0 °C. Se reacciona 0.0500 mol de H⁺ con 0.0500 mol de OH⁻. Valor de referencia: $\Delta_\text{neut}H \approx -55.8$ kJ/mol.

Como el sistema intercambia calor con el ambiente, el aumento de temperatura observado no es el que habría en un calorímetro perfecto ($\Delta T_\text{ad}$). Compararás tres maneras de estimar $\Delta T_\text{ad}$.

**Predicción 7.4.** El método ingenuo, $\Delta T = T_\text{máx} - T_\text{inicial}$, ¿sobreestima o subestima $|\Delta_\text{neut}H|$? ¿Por qué?

In [ ]:
pred_metodo_ingenuo = None      # "sobreestima" o "subestima"

t_s = np.array([
    0, 10, 20, 30, 40, 50, 60, 70, 80, 90,
    100, 110, 120, 130, 140, 150, 160, 170, 180, 190,
    200, 210, 220, 230, 240, 250, 260, 270, 280, 290,
    300, 310, 320, 330, 340, 350, 360, 370, 380, 390,
    400, 410, 420, 430, 440, 450, 460, 470, 480, 490,
    500, 510, 520, 530, 540, 550, 560, 570, 580, 590,
    600,
])
T_C = np.array([
    21.48, 21.54, 21.54, 21.64, 21.66, 21.67, 21.70, 21.75, 21.77, 21.80,
    21.85, 21.87, 21.89, 24.96, 26.47, 27.17, 27.49, 27.60, 27.59, 27.50,
    27.44, 27.37, 27.25, 27.15, 27.06, 26.99, 26.84, 26.79, 26.64, 26.58,
    26.46, 26.43, 26.33, 26.26, 26.14, 26.09, 25.99, 25.92, 25.87, 25.80,
    25.72, 25.64, 25.57, 25.55, 25.47, 25.41, 25.38, 25.26, 25.27, 25.23,
    25.15, 25.08, 25.00, 24.98, 24.90, 24.87, 24.81, 24.78, 24.76, 24.68,
    24.63,
])
t_mezcla, T_amb, C_total, n_rx = 120.0, 23.0, 448.0, 0.0500

plt.plot(t_s, T_C, "o", ms=3)
plt.axvline(t_mezcla, color="k", ls=":", lw=0.8); plt.axhline(T_amb, color="gray", ls="--", lw=0.8)
plt.xlabel("t (s)"); plt.ylabel("T (°C)"); plt.title("Termograma de neutralización"); plt.show()

**Método 1 (ingenuo):** $\Delta T = T_\text{máx} - T(t_\text{mezcla})$, con $T(t_\text{mezcla})$ estimada con una recta ajustada al periodo previo.

**Método 2 (extrapolación lineal):** ajusta una recta al periodo previo ($t\le120$ s) y otra al periodo posterior ($t\ge250$ s); extrapola ambas a $t_\text{mezcla}$ y toma la diferencia.

**Método 3 (balance de energía con ley de enfriamiento de Newton):** si el calor que se pierde hacia el ambiente es proporcional a $T - T_\text{amb}$,
$$\frac{dT}{dt} = \frac{1}{C}\frac{dq_\text{rx}}{dt} - k\,(T - T_\text{amb}) \quad\Longrightarrow\quad \Delta T_\text{ad} = T(t_\text{fin}) - T(0) + k\int_0^{t_\text{fin}} (T - T_\text{amb})\,dt.$$
Estima $k$ como menos la pendiente de $\ln(T - T_\text{amb})$ contra $t$ en el periodo posterior ($t\ge250$ s), cuando la reacción ya terminó.

In [ ]:
pre = t_s <= t_mezcla
post = t_s >= 250
recta_pre = np.polyfit(t_s[pre], T_C[pre], 1)
T_inicial = np.polyval(recta_pre, t_mezcla)

def dH_neut(dT):
    return -C_total*dT/n_rx/1000      # kJ/mol

dT_1 = T_C.max() - T_inicial
dT_2 = None       # TODO: método 2
k_est = None      # TODO: estima k (1/s) del periodo posterior
dT_3 = None       # TODO: método 3 (usa integra)

if pred_metodo_ingenuo is None:
    print("Registra pred_metodo_ingenuo.")
else:
    print(f"{'método':22s} {'ΔT (K)':>8s} {'ΔH (kJ/mol)':>12s} {'|ΔH| vs. ref.':>14s}")
    for nombre, dT in [("1 ingenuo", dT_1), ("2 extrapolación", dT_2), ("3 balance (Newton)", dT_3)]:
        if dT is not None:
            print(f"{nombre:22s} {dT:8.3f} {dH_neut(dT):12.1f} {100*(abs(dH_neut(dT)) - 55.8)/55.8:+8.1f} %")

**Contraste 7.4.**
1. ¿Qué supone cada método sobre el intercambio de calor con el ambiente? ¿Cuál usa más información de los datos?
2. El método 2 aparece en muchos manuales y aquí apenas mejora al 1. Mira la forma del periodo posterior: ¿por qué una recta extrapolada hacia atrás subestima la temperatura en $t_\text{mezcla}$?
3. ¿Por qué el método 3 necesita que $T_\text{amb}$ se mida de forma independiente? ¿Cómo cambia el resultado si $T_\text{amb}$ fuera 22.5 °C? Pruébalo.
4. El método 3 aún difiere un poco del valor de referencia. Propón dos fuentes de esa diferencia.

*Tu respuesta:*

### ★ Ampliación 7.5 — Bomba calorimétrica: del laboratorio a las etiquetas nutrimentales
En una bomba calorimétrica la combustión ocurre a **volumen constante** dentro de un recipiente sellado con O₂ a presión, de modo que el calor medido es $q_V = \Delta U$. Usaremos un **conjunto de datos didáctico simulado**, construido con valores físicamente plausibles. La bomba se calibra quemando ácido benzoico, cuyo $\Delta_c U = -26.434$ kJ/g (patrón certificado).

| muestra | masa (g) | $\Delta T$ (K) |
|---|---|---|
| ácido benzoico | 1.0012 | 2.647 |
| glucosa, C₆H₁₂O₆ | 1.4987 | 2.332 |
| ácido palmítico, C₁₆H₃₂O₂ | 0.6021 | 2.339 |

**Predicción 7.5.** Para la combustión del ácido palmítico, ¿de qué tamaño es $|\Delta_c H - \Delta_c U|/|\Delta_c U|$: ~0.1 %, ~1 % o ~10 %?

**Tu turno.**
1. Calcula $C_\text{bomba}$ (kJ/K).
2. Calcula $\Delta_c U$ de la glucosa y del ácido palmítico en kJ/g y kJ/mol (masas molares: 180.16 y 256.42 g/mol).
3. Escribe las ecuaciones de combustión balanceadas, con H₂O líquida, y obtén $\Delta n_\text{gas}$ en cada caso.
4. Calcula $\Delta_c H = \Delta_c U + \Delta n_\text{gas} RT$ a 298.15 K.
5. Convierte a kcal/g (1 kcal = 4.184 kJ) y compara con los factores de Atwater de las etiquetas: 4 kcal/g para carbohidratos y 9 kcal/g para grasas.

In [ ]:
pred_dif_relativa_palmitico = None     # fracción, p. ej. 0.01

dU_benzoico = -26.434                  # kJ/g
muestras = {   # nombre: (masa g, ΔT K, masa molar g/mol)
    "benzoico":  (1.0012, 2.647, 122.12),
    "glucosa":   (1.4987, 2.332, 180.16),
    "palmitico": (0.6021, 2.339, 256.42),
}
T_ref = 298.15

C_bomba = None                          # TODO (kJ/K)
dn_gas = {"glucosa": None, "palmitico": None}   # TODO

if C_bomba is not None and None not in dn_gas.values():
    print(f"C_bomba = {C_bomba:.3f} kJ/K\n")
    print(f"{'muestra':10s} {'ΔcU kJ/g':>9s} {'ΔcU kJ/mol':>11s} {'Δn_gas':>7s} {'ΔcH kJ/mol':>11s} {'kcal/g':>7s}")
    for nombre in ["glucosa", "palmitico"]:
        m, dT, M = muestras[nombre]
        dU_g = -C_bomba*dT/m
        dU_mol = dU_g*M
        dH_mol = dU_mol + dn_gas[nombre]*R*T_ref/1000
        print(f"{nombre:10s} {dU_g:9.2f} {dU_mol:11.0f} {dn_gas[nombre]:7d} {dH_mol:11.0f} {-dH_mol/M/4.184:7.2f}")
        if nombre == "palmitico":
            revela("7.5", abs(dH_mol - dU_mol)/abs(dU_mol), pred_dif_relativa_palmitico)

**Contraste 7.5.**
1. ¿Por qué para la glucosa $\Delta_c H = \Delta_c U$ sin necesidad de aproximar, mientras que para el ácido palmítico no?
2. Tus valores en kcal/g quedan cerca, pero no exactamente, de 4 y 9. ¿Por qué un factor redondeado para "carbohidratos" o "grasas" no puede coincidir con una molécula particular?
3. Una proteína quemada en la bomba puede liberar ≈5.6 kcal/g, mientras que el factor de Atwater usado para proteínas es menor. Explica **una parte** de la diferencia con el balance de energía del nitrógeno (productos de combustión frente a excreción nitrogenada) y señala al menos otro factor fisiológico, como digestibilidad/absorción o pérdidas metabólicas.
4. En una medición real se corrige por el calor del alambre de ignición y por la formación de HNO₃ a partir del N₂ residual. ¿En qué sentido sesga $\Delta_c U$ cada uno si no se corrige?

*Tu respuesta:*

### ★ Ampliación 7.6 — Calorimetría diferencial de barrido (DSC) de una proteína
El termograma que sigue es **sintético y didáctico**. En un DSC se calienta una disolución de proteína a presión constante y se mide su capacidad calorífica en exceso respecto al amortiguador. Como $q_P = \Delta H$, el **área del pico** de $C_P$ por encima de la línea base es la entalpía de desplegamiento:
$$\Delta H_\text{cal} = \int \left[C_P(T) - C_P^\text{base}(T)\right] dT.$$
El resultado depende de cómo se trace la línea base, y eso no es un detalle técnico: es una decisión con consecuencias numéricas que debe justificarse.

**Tu turno.**
1. **(a)** Usa como línea base la recta del estado nativo ($T\le315$ K) extrapolada a todo el intervalo e integra entre 315 y 345 K.
2. **(b)** Usa como línea base el segmento que une la recta nativa evaluada en 315 K con la recta desplegada ($T\ge345$ K) evaluada en 345 K.
3. Repite (b) moviendo los límites de integración ±5 K. Reporta $\Delta H_\text{cal}$ con una incertidumbre **sistemática** debida a la línea base y compárala con la que causaría el ruido.
4. ¿Por qué la línea base después de la transición queda por arriba de la de antes? ¿Qué significado físico tiene $\Delta C_P > 0$ al desplegar una proteína?

In [ ]:
T_K = np.arange(300, 360.01, 0.5)
Cp_kJ = np.array([      # kJ/(mol·K)
    1.60, 1.63, 1.46, 1.68, 2.58, 1.84, 2.19, 1.62, 2.19, 1.44,
    1.87, 3.07, 1.95, 2.65, 2.40, 2.17, 1.99, 2.78, 2.21, 2.91,
    2.03, 2.36, 2.54, 2.53, 1.91, 2.17, 2.38, 2.16, 2.81, 2.82,
    1.86, 1.98, 2.27, 2.93, 3.81, 2.99, 3.42, 3.11, 3.39, 3.35,
    4.54, 5.44, 5.54, 7.02, 7.48, 8.54, 10.16, 11.83, 14.09, 15.66,
    17.79, 21.83, 24.97, 28.90, 33.43, 36.16, 40.51, 43.73, 45.97, 47.77,
    47.64, 46.99, 46.11, 43.56, 40.78, 36.86, 33.80, 29.44, 25.69, 22.13,
    19.65, 17.91, 14.83, 13.38, 12.85, 11.04, 10.33, 8.33, 7.01, 8.00,
    6.34, 6.55, 6.28, 5.55, 6.35, 6.18, 5.65, 5.31, 5.32, 4.74,
    5.04, 5.05, 4.73, 5.39, 4.46, 4.85, 4.12, 5.22, 5.32, 5.28,
    6.20, 5.42, 4.68, 6.13, 4.71, 5.40, 5.40, 4.76, 5.69, 5.28,
    5.38, 4.90, 5.70, 4.85, 4.50, 5.80, 5.07, 5.27, 6.04, 5.74,
    5.55,
])
plt.plot(T_K, Cp_kJ, ".", ms=4)
plt.xlabel("T (K)"); plt.ylabel("$C_P$ en exceso (kJ mol$^{-1}$ K$^{-1}$)"); plt.title("Termograma DSC"); plt.show()

In [ ]:
def dH_dsc(T1=315.0, T2=345.0, metodo="b"):
    nat = np.polyfit(T_K[T_K <= T1], Cp_kJ[T_K <= T1], 1)
    des = np.polyfit(T_K[T_K >= T2], Cp_kJ[T_K >= T2], 1)
    m = (T_K >= T1) & (T_K <= T2)
    # TODO: construye la línea base según 'metodo' ("a" o "b") sobre T_K[m] y devuelve la integral (kJ/mol)
    return None

if dH_dsc() is not None:
    print(f"(a) línea base nativa extrapolada: ΔH = {dH_dsc(metodo='a'):.0f} kJ/mol")
    print(f"(b) línea base recta nativa→desplegada: ΔH = {dH_dsc(metodo='b'):.0f} kJ/mol")
    # TODO: barrido de límites ±5 K para el método (b)

## 8. Problemas de síntesis (elige dos)
Para cada problema entrega: (i) planteamiento y supuestos, (ii) estimación de orden de magnitud **antes** de programar, (iii) cálculo en código, (iv) evaluación crítica: ¿qué supuesto cambiaría más el resultado y en qué dirección?

**A. Fisiología.** El metabolismo basal de un adulto es de ~80–100 W. Si el cuerpo (70 kg, $c\approx3.5$ J/(g·K)) fuera adiabático, ¿cuánto subiría su temperatura en una hora? Si toda esa energía se disipara evaporando sudor ($\Delta_\text{vap}H\approx2.41$ kJ/g a 37 °C), ¿cuánta agua haría falta por día? Compara con datos de pérdida de agua que encuentres en una fuente confiable y explica la diferencia.

**B. Inflador de bicicleta.** Aire a 1 atm y 298 K se comprime adiabática y reversiblemente hasta 5 atm ($\gamma=1.4$). ¿Qué temperatura final predice el modelo? La temperatura medida en la pared de una bomba real suele ser mucho menor que la temperatura adiabática máxima predicha para el gas y depende de la rapidez del bombeo: ¿qué supuestos del modelo ideal fallan y cómo los modificarías?

**C. Diseño de un ciclo.** Diseña un ciclo de al menos tres tramos de tipos distintos para 1 mol de gas monoatómico, con $50 \le P \le 300$ kPa y $5 \le V \le 40$ L, que funcione como motor con $w_\text{neto} = -1000 \pm 20$ J. Verifica con `analiza_ciclo` y `dibuja_ciclo`, e identifica en qué tramos entra calor.

**D. Crítica.** Un compañero afirma: *"Si una reacción en disolución tiene $\Delta H<0$, la disolución siempre se calienta."* ¿En qué condiciones experimentales es falso? (Piensa en un calorímetro de titulación isotérmica, ITC: ¿qué mide si la temperatura de la celda se mantiene constante?)

In [ ]:
# Espacio de trabajo para tus problemas de síntesis

## 9. Cierre
1. **Regresa al diagnóstico (§0).** Para cada enunciado, escribe tu respuesta actual y, si cambió, qué sección del cuaderno te hizo cambiarla.
2. **Una pregunta abierta.** Escribe una pregunta que te haya surgido y que este cuaderno no responda.

### Qué se entrega

Entrega **únicamente las secciones asignadas por el docente**. Como mínimo, cada bloque evaluado debe conservar:

- la predicción inicial sin corregir retrospectivamente;
- el planteamiento termodinámico y los supuestos;
- las celdas `# TODO` correspondientes a la ruta asignada, **o la alternativa accesible indicada por el docente**;
- la verificación independiente y el contraste;
- cuando se haya usado IA, una entrada breve de bitácora PIVED;
- los problemas de síntesis que se hayan asignado.

No es requisito completar todas las ampliaciones ★.

### Criterios de evaluación sugeridos

| criterio | peso |
|---|---:|
| Planteamiento y razonamiento termodinámico | 30 % |
| Predicciones y contrastes: calidad del razonamiento, no acierto | 20 % |
| Verificación, cacería de errores y control de unidades/signos | 20 % |
| Interpretación de gráficas/código o representación equivalente | 15 % |
| Síntesis y capacidad de defender conclusiones | 15 % |

El código se valora como medio de modelado cuando forma parte del objetivo de la actividad; no se premia la complejidad del código por sí misma.
